## Dynamic Data Analysis - Streaming (35 points)

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.streaming import DataStreamWriter
import time

In [0]:
SCHEMA = "device_id STRING, event_date INT, event_time INT, station_num STRING, prog_code STRING, household_id STRING"
kafka_server = "kafka.eastus.cloudapp.azure.com:29092"
topic = "view_data" 
OFFSETS_PER_TRIGGER = 50000

streaming_df = spark.readStream\
                  .format("kafka")\
                  .option("kafka.bootstrap.servers", kafka_server)\
                  .option("subscribe", topic)\
                  .option("startingOffsets", "earliest")\
                  .option("failOnDataLoss",False)\
                  .option("maxOffsetsPerTrigger", OFFSETS_PER_TRIGGER)\
                  .load()\
                  .select(from_csv(decode("value", "US-ASCII"), schema=SCHEMA).alias("value")).select("value.*")

In [0]:
import pyspark.sql.functions as F

# Global counters and accumulators
batch_count = 0
MIN_BATCHES_TO_PROCESS = 3
max_rows = 50

# Cumulative state dictionaries
cumulative_subset_counts = {}
cumulative_general_counts = {}

def process_batch(df, epoch_id):
    global batch_count, cumulative_subset_counts, cumulative_general_counts

    batch_count += 1

    if df.isEmpty():
        print(f"Batch {batch_count} is empty. Skipping analysis.")
        if batch_count >= MIN_BATCHES_TO_PROCESS:
            try:
                print(f"Processed at least {MIN_BATCHES_TO_PROCESS} batches. Stopping stream.")
                query.stop()
            except Exception as e:
                print(f"Caught exception while stopping (expected during shutdown): {e}")
        return

    # Join with clustering info
    batch_with_cluster_info = df.join(
        broadcast(clustered_df.select("household_id", "cluster", "subset_3rds")),
        on="household_id",
        how="inner"
    )

    # Subset: only "3rds"
    subset_df = batch_with_cluster_info.filter(col("subset_3rds") == True)

    # Subset counts per cluster/station
    subset_counts = subset_df.groupBy("cluster", "station_num").agg(F.count("*").alias("subset_count"))

    # General counts per cluster/station
    general_counts = batch_with_cluster_info.groupBy("cluster", "station_num").agg(F.count("*").alias("general_count"))

    # Update cumulative counts (subset)
    for row in subset_counts.collect():
        key = (row['cluster'], row['station_num'])
        cumulative_subset_counts[key] = cumulative_subset_counts.get(key, 0) + row['subset_count']

    # Update cumulative counts (general)
    for row in general_counts.collect():
        key = (row['cluster'], row['station_num'])
        cumulative_general_counts[key] = cumulative_general_counts.get(key, 0) + row['general_count']

    # Compute cumulative totals per cluster
    cumulative_subset_totals = {}
    cumulative_general_totals = {}

    for (cluster, _), count in cumulative_subset_counts.items():
        cumulative_subset_totals[cluster] = cumulative_subset_totals.get(cluster, 0) + count

    for (cluster, _), count in cumulative_general_counts.items():
        cumulative_general_totals[cluster] = cumulative_general_totals.get(cluster, 0) + count

    # Build cumulative diff_rank list
    rows = []
    for (cluster, station) in cumulative_subset_counts:
        subset_count = cumulative_subset_counts[(cluster, station)]
        general_count = cumulative_general_counts.get((cluster, station), 0)

        subset_total = cumulative_subset_totals[cluster]
        general_total = cumulative_general_totals.get(cluster, 0)

        subset_pct = (subset_count / subset_total) * 100 if subset_total else 0
        general_pct = (general_count / general_total) * 100 if general_total else 0

        diff = subset_pct - general_pct
        rows.append((cluster, station, diff))

    # Convert to DataFrame
    diff_rank_df = spark.createDataFrame(rows, ["cluster", "station_num", "diff_rank"])

    # Top 7 stations per cluster by cumulative diff_rank
    window_spec = Window.partitionBy("cluster").orderBy(col("diff_rank").desc())
    top_diff_rank = diff_rank_df.withColumn("rank", rank().over(window_spec)) \
        .filter(col("rank") <= 7) \
        .select("cluster", "station_num", "diff_rank") \
        .orderBy("cluster", col("diff_rank").desc())

    print(f"Top Stations per Cluster by cumulative diff_rank (3rds subset) - Batch {batch_count}:")
    top_diff_rank.show(n=max_rows, truncate=False)

    if batch_count >= MIN_BATCHES_TO_PROCESS:
        try:
            query.stop()
        except Exception as e:
            pass


# Start the streaming query
query = streaming_df.writeStream \
    .outputMode("append") \
    .foreachBatch(process_batch) \
    .start()

print("Streaming query started. Waiting for batches...")

query.awaitTermination()
print("Streaming process complete.")

Streaming query started. Waiting for batches...
Top Stations per Cluster by cumulative diff_rank (3rds subset) - Batch 1:
+-------+-----------+-------------------+
|cluster|station_num|diff_rank          |
+-------+-----------+-------------------+
|0      |32645      |0.6545567491111535 |
|0      |11150      |0.3663890507698895 |
|0      |10142      |0.3397109784317117 |
|0      |11164      |0.3289651967812214 |
|0      |11158      |0.3103270179764027 |
|0      |10057      |0.3024346208220746 |
|0      |10021      |0.25522773427513623|
|1      |14771      |0.4296431285528839 |
|1      |12131      |0.32321660378825023|
|1      |10145      |0.30635116359920533|
|1      |11713      |0.28078792970966093|
|1      |70522      |0.24087692685278   |
|1      |16300      |0.23690187883327057|
|1      |25544      |0.2118497828930999 |
|2      |10647      |0.7296580251695254 |
|2      |12574      |0.6615144294277733 |
|2      |10731      |0.628596883178621  |
|2      |18480      |0.496926698182010